In [ ]:
# Imports
#%matplotlib widget
from ufpdf_xfel_scripts.lcls.paths import synchrotron_data_dir, experiment_data_dir, global_output_dir
from ufpdf_xfel_scripts.lcls.run import Run
from ufpdf_xfel_scripts.lcls.plots import (plot_delay_scans,
                                           plot_static_scans,
                                           plot_fq_delay_scans,
                                           plot_reference_comparison,
                                           plot_gr_function,
                                           plot_time_resolved_window_map,
                                           plot_morph_parameters)

In [ ]:
# Experiment: LCLS 202601
instrument = 'mfx'
experiment_number = 'l1044925'

In [ ]:
# User: Luis
username = 'lkitsu'

In [ ]:
# Setup:

# azimuthal sectors
vertical = [0,6,12]
horizontal = [2,3,9,10]
total = [0,1,2,3,4,5,6,7,8,9,10,11,12]

# setup for roi
q_min =8 # I(q) figure of merit (fom)
q_max =12 # I(q) fom
r_min_fom = 6 # G(r) fom 
r_max_fom = 7 # G(r) fom

# global setup for morphs
target = "average_off" # morph target: "single_delay" or "average_off"
target_id = 0 # index of the delay_time Iq_off to use as target for morphing as well as calculate adhoc corrections for g(r)

# setup for morph I(q) delay
q_min_morph = 0 # qmin for normalization. None uses the smaller value. Not working for new diffpy.morph! Also is called xmin now.
q_max_morph = 14 # qmax for normalization. None uses the largest value. Not working for new diffpy.morph! Also is called xmax now.
delay_scale = 1 #
delay_hshift = 0.01
delay_vshift = None
delay_stretch = None
delay_smear = None
delay_baselineslope = None
delay_qdamp = None
t0 = 0
points_away_t0_plot_on_off = 0

# setup for morph F(q)
fit_qmin = 0.1
fit_qmax = 11
pdf_rmin = 0
pdf_rmax = 30
getx_scale = 1
getx_squeeze_parms = [0,0,0,0,0]

# setup for pdfgetx3
pdfgetter_config = {
    "rpoly": 1.2,
    "qmin": 0.5,
    "qmax": 11.,
    "qmaxinst": 11.5,
    "bgscale": 0,     # set 0 to suppress background correction
}

In [ ]:
# Run:
run_number = 111
background_number = 1
sample_name = 'NSWS'
sample_composition = 'Na11SnPS12'
azimuthal_selector = total # List of integers, e.g., [1,6,13]. Pre-defined options are vertical, horizontal, total.
number_of_static_samples = 11   # only needed for static runs, will be ignored otherwise
verbose = False
plot_monitor_normalizations = False
# delay motor name used for this run
delay_motor = "mfx_lxt_fast2"
#delay_motor = "lxt_ttx"

In [ ]:
# Instantiate run object
run_object = Run(run_number,
                 background_number,
                 sample_name,
                 sample_composition,
                 instrument,
                 experiment_number,
                 number_of_static_samples=number_of_static_samples,
                 delay_motor=delay_motor,
                 delay_scale=delay_scale,
                 delay_hshift=delay_hshift,
                 delay_vshift=delay_vshift,
                 delay_stretch=delay_stretch,
                 delay_smear=delay_smear,
                 pdfgetter_config=pdfgetter_config,
                 getx_scale=getx_scale,
                 getx_squeeze_parms=getx_squeeze_parms,
                 q_min=q_min,
                 q_max=q_max,
                 fit_qmin=fit_qmin,
                 fit_qmax=fit_qmax,
                 q_min_morph=q_min_morph,
                 q_max_morph=q_max_morph,
                 pdf_rmin=pdf_rmin,
                 pdf_rmax=pdf_rmax,
                 azimuthal_selector=azimuthal_selector,
                 target=target,
                 )
morph_parameters = run_object.morph_parameters # Dictionary with fitted morph parameters. The key is the delay.

In [ ]:
# Plot Iq
if run_object.delay_scan:
    plot_delay_scans(run_object.morphed_delay_scans, run_object)
    if plot_monitor_normalizations:
        plot_delay_scans(run_object.mon1_normalized_delay_scans, run_object)
        plot_delay_scans(run_object.mon2_normalized_delay_scans, run_object)
else:
    plot_static_scans(run_object.morphed_static_scans, run_object)


In [ ]:
plot_morph_parameters(run_object.morph_parameters)

In [ ]:
# Plot f(Q)
if run_object.delay_scan:
    plot_fq_delay_scans(run_object.fq_delay_scans, run_object)

In [ ]:
reference_delay = run_object.target_delay
q_morph = run_object.fq_delay_scans[reference_delay]["q"]
fq_morph = run_object.fq_delay_scans[reference_delay]["fq_off"]  # or fq_on
plot_reference_comparison(
    q_target=run_object.q_synchrotron,
    fq_target=run_object.fq_synchrotron,
    q_morph=q_morph,
    fq_morph=fq_morph,
    r_min=pdf_rmin,
    r_max=pdf_rmax,
)

In [ ]:
plot_gr_function(
    run_object.gr_delay_scans,
    sample_name=sample_name,
    run_number=run_number,
    r_min_fom=r_min_fom,
    r_max_fom=r_max_fom
)

In [ ]:
plot_time_resolved_window_map(
    run_object.morphed_delay_scans,
    run_object,
    width=0.3,
    n_centers=200,
    metric="rms", # possible options "rms"-> sqrt(mean(diff^2)), "l1"-> mean(|diff|), "mean"-> mean(diff), "peak"-> max(|diff|), "var"-> variance(diff)
)

In [ ]:
plot_time_resolved_window_map(
    run_object.fq_delay_scans,
    run_object,
    width=0.3,
    n_centers=200,
    metric="l1", # possible options "rms"-> sqrt(mean(diff^2)), "l1"-> mean(|diff|), "mean"-> mean(diff), "peak"-> max(|diff|), "var"-> variance(diff)
)

In [ ]:
plot_time_resolved_window_map(
    run_object.gr_delay_scans,
    run_object,
    width=0.2,
    n_centers=200,
    metric="l1", # possible options "rms"-> sqrt(mean(diff^2)), "l1"-> mean(|diff|), "mean"-> mean(diff), "peak"-> max(|diff|), "var"-> variance(diff)
)